In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[1]
DATASET_DIR = PROJECT_ROOT/"Dataset"
MODEL_DIR = PROJECT_ROOT/"models"
IR_DIR = PROJECT_ROOT/"Pz and Sz"
FIG_DIR = PROJECT_ROOT/"figs"
print(PROJECT_ROOT)
print(DATASET_DIR)
print(MODEL_DIR)
print(FIG_DIR)

/home/rkobayashi/anc/GFANC-custom
/home/rkobayashi/anc/GFANC-custom/Dataset
/home/rkobayashi/anc/GFANC-custom/models
/home/rkobayashi/anc/GFANC-custom/figs


In [2]:
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [3]:

import matplotlib.pyplot as plt
import numpy as np
import torch

from common.loading_real_wave_noise import loading_real_wave_noise
from common.Reading_path_test import loading_paths_from_MAT
from data.Disturbance_generation import Disturbance_generation_from_real_noise
from fxlms.FxLMS_algorithm_v2 import FxLMS, train_fxlms_algorithm
from gfanc.Control_filter_selection_pre_now import Control_filter_selection_pre_now
from gfanc.Fixed_filter_noise_cancellation_subfilters import My_Fixed_filter_controller

print(torch.cuda.is_available())

True


In [4]:
# Loading real noises
mdict = {}
fs = 16000
StepSize = 0.0001
sound_name = 'Helicopter'
waveform, resample_rate = loading_real_wave_noise(folde_name=PROJECT_ROOT/'Real Noise Examples/', sound_name=sound_name+'.wav')
waveform = waveform[0].unsqueeze(0)

In [5]:

# Loading path
subfolder = "Dongyuan"
Pri_path, Secon_path = loading_paths_from_MAT(folder=IR_DIR, subfolder=subfolder, Pri_path_file_name='Primary_path.mat', Sec_path_file_name='Secondary_path.mat')
Dis, Fx, Re = Disturbance_generation_from_real_noise(fs=fs, Repet=0, wave_form=waveform, Pri_path=Pri_path, Sec_path=Secon_path)
# Dis: disturbance (cotrolled noise)， Fx: fixed-x signal, Re: repeated waveform (primary_noise) Repetition=Repet+1
import matplotlib as mpl

mpl.rcParams['agg.path.chunksize'] = 10000

In [6]:
def additional_noise(signal, snr_db):
    signal_power = signal.square().mean()
    additional_noise = torch.randn_like(signal, dtype=signal_power.dtype, device=signal_power.device)
    snr = 10 ** (snr_db / 10)
    noise_power = signal_power / snr
    return signal + noise_power.sqrt() * additional_noise

In [7]:
ErrorFixeds = {}
ErrorFxLMSs = {}

In [ ]:
snrs = [20, 15, 10, 5, 2, 1, 0, -1]
# GFANC: combining previous and present frame to predict the next

MODEL_PTH = MODEL_DIR/"M6_res_Synthetic.pth"
path_mat = MODEL_DIR/'Pretrained_Sub_Control_filters.mat'
for snr in snrs:
    Ref = additional_noise(Re, snr)
    Filter_vector = Control_filter_selection_pre_now(fs=16000, MODEL_PTH=MODEL_PTH, path_mat=path_mat, Primary_noise=Ref.unsqueeze(0), threshold=0.5)

    Fixed_Cancellation = My_Fixed_filter_controller(Filter_vector=Filter_vector, fs=16000, Secondary_path=torch.tensor(Secon_path, dtype=torch.float32))
    ErrorFixed = Fixed_Cancellation.noise_cancellation(Dis=Dis, Ref=Ref)
    ErrorFixeds[snr] = 10*torch.log10(ErrorFixed.square().mean()/Dis.square().mean())
print(ErrorFixeds)

tensor(0.0551)
tensor(0.0006)
The primary nosie has 10 seconds !!!
1 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
2 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
3 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
4 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
5 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
6 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
7 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
8 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
9 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
10 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
tensor(0.0551)
tensor(0.0017)
The primary nosie has 10 seconds !!!
1 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
2 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
3 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
4 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
5 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
6 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
7 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
8 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
9 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
10 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
tensor(0.0551)
tensor(0.0055)
The primary nosie has 10 seconds !!!
1 [1 1 1 0 0 0 0 0 0 0 1 0 0 0 0]
2 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
3 [1 1 1 1 0 0 0 0 0 0 1 0 0 0 0]
4 [1 1 1 1 0 0 

In [ ]:
# FxLMS
snrs = [20, 10, 5, 2.5, 1, 0, -1]
StepSizes = [ StepSize * 10 ** i for i in range(4)]
for snr in snrs:
    ErrorTemp = {}
    _, Noisy_Fx, _ = Disturbance_generation_from_real_noise(fs=fs, Repet=0, wave_form=waveform, Pri_path=Pri_path, Sec_path=Secon_path)
    for stepsize in StepSizes:
        controller = FxLMS(Len=1024) # 1024 is the same size of coeffient vector of fixed-filter
        ErrorFxLMS = train_fxlms_algorithm(Model=controller, Ref=Noisy_Fx, Disturbance=Dis, Stepsize=stepsize) # torch.Size([320000])
        ErrorTemp[stepsize] = 10*torch.log10(torch.tensor(ErrorFxLMS).square().sum()/Dis.square().sum())
    ErrorFxLMSs[snr] = ErrorTemp
print(ErrorFxLMSs)


[========================================================================] 100%


tensor(-5.1006)


[========================================================================] 100%


tensor(-11.0309)


[========================================================================] 100%


...

tensor(-17.7002)


[========================================================================] 100%


tensor(-23.3256)


[========================================================================] 100%


tensor(nan)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# 横軸の位置を等間隔にする
delays = [-25, 0, 5]
x_positions = range(len(delays))

for param, values in ErrorFxLMSs.items():
    y = [float(values[d]) if d in values else float("nan")
         for d in delays]

    ax.plot(
        x_positions[:len(delays)],
        y,
        marker="o",
        linewidth=2,
        label=f"StepSize={param}",
    )

# GFANC
y = [float(ErrorFixeds[d]) for d in delays]

ax.plot(
    x_positions,
    y,
    marker="s",
    linestyle="--",
    linewidth=2,
    label="GFANC",
)

ax.set_xlabel("Delay")
ax.set_ylabel("NMSE")
ax.set_title("Compare GFANC with FxLMS")

# 等間隔の位置に元の遅延値を表示
ax.set_xticks(list(x_positions))
ax.set_xticklabels(delays)

ax.grid(True, linestyle=":", alpha=0.7)
ax.legend()
fig.tight_layout()
plt.show()